# AraBridge: QLoRA Fine-Tuning for Arabic Dialect Identification

This notebook fine-tunes a dialect identification model using QLoRA (Quantized Low-Rank Adaptation)
on the MADAR dataset. Designed to run on Google Colab with a free T4 GPU.

## Overview
- **Base Model**: `aubmindlab/bert-base-arabertv2`
- **Technique**: 4-bit QLoRA with PEFT
- **Task**: Arabic Dialect Classification (MSA, Gulf, Egyptian, Levantine)
- **Target**: UAE AI job portfolio — demonstrates Arabic NLP fine-tuning expertise

## Prerequisites
- Google Colab with GPU runtime (T4 is sufficient)
- Hugging Face account (for model access)

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes torch scikit-learn wandb

## 2. Imports & Configuration

In [ ]:
import os
import torch
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import json

# Configuration
MODEL_NAME = "aubmindlab/bert-base-arabertv2"
OUTPUT_DIR = "./arabert-dialect-qlora"
NUM_EPOCHS = 5
BATCH_SIZE = 8
LEARNING_RATE = 2e-4
MAX_LENGTH = 128

# Dialect labels
LABELS = ["MSA", "Gulf", "Egyptian", "Levantine"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

print(f"Using GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"Labels: {LABELS}")

## 3. Prepare Training Data

MADAR (Multi-Arabic Dialect Applications and Resources) is the standard benchmark for Arabic dialect identification.
We use a curated set of dialect samples across MSA, Gulf, Egyptian, and Levantine.

In [ ]:
# Sample training data — MADAR-style dialect examples
# In production, you'd load the full MADAR corpus from:
# https://camel.abudhabi.nyu.edu/madar-parallel-corpus/

TRAINING_DATA = [
    # MSA
    {"text": "اللغة العربية الفصحى هي اللغة الرسمية في العديد من الدول", "label": "MSA"},
    {"text": "تسعى الحكومة إلى تطوير قطاع التعليم", "label": "MSA"},
    {"text": "الاقتصاد العالمي يشهد تغيرات كبيرة في الوقت الحالي", "label": "MSA"},
    {"text": "أشار التقرير إلى وجود تحسن ملحوظ في الأداء", "label": "MSA"},
    {"text": "ينبغي على الجميع الالتزام بالقوانين والأنظمة", "label": "MSA"},
    {"text": "الأمم المتحدة تدعو إلى السلام في المنطقة", "label": "MSA"},
    {"text": "الدراسة تؤكد أهمية البحث العلمي للتقدم", "label": "MSA"},
    {"text": "المؤتمر سيعقد في العاصمة الأسبوع القادم", "label": "MSA"},
    {"text": "تم الإعلان عن نتائج المسابقة الرسمية", "label": "MSA"},
    {"text": "الكتاب يتناول موضوعا مهما في التاريخ", "label": "MSA"},
    
    # Gulf
    {"text": "شو أخبارك يالغالي عساك طيب", "label": "Gulf"},
    {"text": "ما عندي مانع أروح وياهم الحين", "label": "Gulf"},
    {"text": "ترى الموضوع سهل ترى ما يحتاج", "label": "Gulf"},
    {"text": "شكثر سعر هالشي إذا ما عليك أمر", "label": "Gulf"},
    {"text": "إي والله الأمور طيبة والحمدلله", "label": "Gulf"},
    {"text": "وين رايح الحين خلنا نتمشى شوي", "label": "Gulf"},
    {"text": "خلنا نشوف شو يصير عقب", "label": "Gulf"},
    {"text": "هذا شي زين ما شاء الله", "label": "Gulf"},
    {"text": "بس خلاص تعبت من هالوضع", "label": "Gulf"},
    {"text": "هلا والله حياك الله يالغالي", "label": "Gulf"},
    
    # Egyptian
    {"text": "إزيك عامل إيه النهاردة", "label": "Egyptian"},
    {"text": "عايز أروح السينما معاك بكرة", "label": "Egyptian"},
    {"text": "الموضوع ده ممل أوي مش قادر", "label": "Egyptian"},
    {"text": "أنا مش فاهم حاجة خالص", "label": "Egyptian"},
    {"text": "إيه ده بجد أنا مصدوم", "label": "Egyptian"},
    {"text": "أنا جاي لك بكرة إن شاء الله", "label": "Egyptian"},
    {"text": "دي حاجة جميلة قوي ما شاء الله", "label": "Egyptian"},
    {"text": "كده كده هنشوف اللي يحصل", "label": "Egyptian"},
    {"text": "أنا تعبان النهاردة مش طايق حاجة", "label": "Egyptian"},
    {"text": "يا عم الحاج إنت فين من زمان", "label": "Egyptian"},
    
    # Levantine
    {"text": "شو بدك تاكل اليوم", "label": "Levantine"},
    {"text": "كيفك يا حبيبي شو الأخبار", "label": "Levantine"},
    {"text": "بدي روح عالسوق أشتري شوي", "label": "Levantine"},
    {"text": "شو هالحكي ما فهمت عليك", "label": "Levantine"},
    {"text": "له يا زلمة شو هالحركات", "label": "Levantine"},
    {"text": "وين رايح هلا بدي إجيك", "label": "Levantine"},
    {"text": "عم دور على شغل هالأيام", "label": "Levantine"},
    {"text": "خلصت الشغل اليوم الحمدلله", "label": "Levantine"},
    {"text": "ما بدي أحكي كتير خلينا نشوف", "label": "Levantine"},
    {"text": "شو رأيك نطلع نتمشى شوي", "label": "Levantine"},
]

# Create Dataset
texts = [item["text"] for item in TRAINING_DATA]
labels = [LABEL2ID[item["label"]] for item in TRAINING_DATA]

# Split: 80% train, 20% eval
import random
random.seed(42)
indices = list(range(len(texts)))
random.shuffle(indices)

split = int(0.8 * len(indices))
train_indices = indices[:split]
eval_indices = indices[split:]

dataset = DatasetDict({
    "train": Dataset.from_dict({
        "text": [texts[i] for i in train_indices],
        "label": [labels[i] for i in train_indices],
    }),
    "eval": Dataset.from_dict({
        "text": [texts[i] for i in eval_indices],
        "label": [labels[i] for i in eval_indices],
    }),
})

print(f"Train samples: {len(dataset['train'])}")
print(f"Eval samples: {len(dataset['eval'])}")
print(f"\nSample: {dataset['train'][0]}")

## 4. Load Tokenizer & Tokenize Data

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset.set_format("torch")

print(f"Tokenized train: {tokenized_dataset['train']}")
print(f"Tokenized eval: {tokenized_dataset['eval']}")

## 5. Load Model with 4-bit Quantization

In [ ]:
# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load base model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    quantization_config=bnb_config,
    device_map="auto",
)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r=16,               # rank
    lora_alpha=32,       # scaling
    target_modules=["query", "key", "value", "output.dense"],  # AraBERT attention modules
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"Model loaded. Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 6. Define Metrics

In [ ]:
def compute_metrics(eval_pred):
    """Compute accuracy and F1 score for evaluation."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')

    return {
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
    }

## 7. Training

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    learning_rate=LEARNING_RATE,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=True,
    report_to="none",  # Set to "wandb" if using Weights & Biases
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["eval"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting training...")
trainer.train()

## 8. Evaluate

In [ ]:
# Evaluate on test set
eval_results = trainer.evaluate()
print("\n" + "="*50)
print("EVALUATION RESULTS")
print("="*50)
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

# Generate predictions for analysis
predictions = trainer.predict(tokenized_dataset["eval"])
pred_labels = np.argmax(predictions.predictions, axis=-1)

print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(
    tokenized_dataset["eval"]["label"].cpu(),
    pred_labels,
    target_names=LABELS,
))

## 9. Save & Export Model

In [ ]:
# Save the LoRA adapter
adapter_path = f"{OUTPUT_DIR}/lora-adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"LoRA adapter saved to: {adapter_path}")

# Merge and save full model (optional — larger but easier to deploy)
print("\nMerging LoRA weights into base model...")
merged_model = model.merge_and_unload()

merged_path = f"{OUTPUT_DIR}/merged-model"
merged_model.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)

print(f"Merged model saved to: {merged_path}")
print("\n✅ Ready for deployment with AraBridge!")

## 10. Test Inference

Test the fine-tuned model on sample sentences from each dialect.

In [ ]:
# Reload and test
from peft import PeftModel, PeftConfig

test_sentences = [
    ("تسعى الحكومة إلى تطوير التعليم العالي", "MSA"),
    ("شو أخبارك يالغالي عساك بخير", "Gulf"),
    ("إزيك عامل إيه النهاردة يا صاحبي", "Egyptian"),
    ("شو بدك تاكل اليوم يا زلمة", "Levantine"),
]

print("Testing fine-tuned model:")
print("-" * 60)

for text, expected in test_sentences:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]
        pred_idx = torch.argmax(probs).item()
        pred_label = ID2LABEL[pred_idx]
        confidence = probs[pred_idx].item()

    status = "✅" if pred_label == expected else "❌"
    print(f"{status} Text: {text}")
    print(f"   Expected: {expected} | Predicted: {pred_label} ({confidence:.2%})")
    print()

print("-" * 60)
print("🎉 Fine-tuning complete! Model ready for AraBridge integration.")

## Next Steps

1. **More Data**: Load the full MADAR corpus for better accuracy
2. **Hyperparameter Tuning**: Experiment with LoRA rank, learning rate, batch size
3. **Evaluation**: Test on MADAR test set and compare with published baselines
4. **Deploy**: Copy the merged model to AraBridge's `backend/` and reference it in `config.py`
5. **Track**: Log experiments to Weights & Biases (set `report_to="wandb"`)

### MADAR Dataset Access
```python
# Full MADAR corpus is available via CAMeL Lab:
# https://camel.abudhabi.nyu.edu/madar-parallel-corpus/
#
# Or via Hugging Face:
# from datasets import load_dataset
# madar = load_dataset("madar", "all")
```